# Corner plot explorer

Plain version: edit the `PULSAR` / `CASES_ON` variables in the cell below and re-run it -- no
checkboxes, no widget state to get confused by. Each edit + run gives you a fresh pair of corner
plots (rednoise, then dmgp if any selected case has one) right below the cell.

**Run this on the cluster**, from the repo root: needs `samples/<dataset>/<pulsar>/<model>/` (the
raw h5 chains, cluster-only, never pulled locally). Login node `python3.9` kernel, already has
numpy/matplotlib/corner/h5py/scipy (user-installed, `python3.9 -m pip install --user ...`).

In [ ]:
import os
# Jupyter/VS Code sets the kernel's cwd to the notebook's own folder by
# default -- everything below assumes cwd == repo root (where scripts/,
# feathers_*/, samples/ etc. live), so step up out of notebooks/ if needed.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
assert os.path.isdir("scripts"), f"expected repo root (with a scripts/ dir) as cwd, got {os.getcwd()}"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
sys.path.insert(0, os.path.join(os.getcwd(), "scripts", "utils"))

import numpy as np
import matplotlib.pyplot as plt
import corner as corner_module
from matplotlib.lines import Line2D

from compare_posteriors import load_run, CASE_STYLE
from plot_posteriors import group_of
import pyarrow.feather as feather

# compare_posteriors.py calls matplotlib.use("Agg") at import time (it's a
# CLI script meant to run non-interactively, just saving PDFs to disk) --
# that global backend setting leaks into this kernel too, since it's the
# same matplotlib module object, and silently breaks inline figure display
# here. Switch back to the interactive inline backend now that the import
# is done.
get_ipython().run_line_magic("matplotlib", "inline")

PULSARS = ["J1012+5307", "J1640+2224", "J1744-1134"]
SAMPLES_DIR_BASE = "samples"

In [ ]:
# burn_epochs per (dataset, model, pulsar) -- the values already established
# for this project (see memory/burnin_epochs_dmcorrection.md); same ones the
# compare_posteriors.py CLI calls use. Keyed by model too, not just dataset+
# pulsar: J1012+5307's combined_corrected needs burn=19 for rnonly but only
# 12 for dmrn (reviewed directly from each one's own trace). Edit here if a
# run gets re-burned.
BURN_EPOCHS = {}
for p in PULSARS:
    BURN_EPOCHS[("feathers_highfreq_corrected", "dmrn", p)] = 4
    BURN_EPOCHS[("feathers_highfreq_corrected", "rnonly", p)] = 4
    BURN_EPOCHS[("feathers_highfreq_uncorrected", "rnonly", p)] = 4

BURN_EPOCHS.update({
    ("feathers_combined_corrected", "dmrn",   "J1012+5307"): 12,
    ("feathers_combined_corrected", "rnonly", "J1012+5307"): 19,
    ("feathers_combined_corrected", "dmrn",   "J1640+2224"): 4,
    ("feathers_combined_corrected", "rnonly", "J1640+2224"): 4,
    ("feathers_combined_corrected", "dmrn",   "J1744-1134"): 16,
    ("feathers_combined_corrected", "rnonly", "J1744-1134"): 16,

    ("feathers_combined_uncorrected", "dmrn",   "J1012+5307"): 8,
    ("feathers_combined_uncorrected", "rnonly", "J1012+5307"): 8,
    ("feathers_combined_uncorrected", "dmrn",   "J1640+2224"): 4,
    ("feathers_combined_uncorrected", "rnonly", "J1640+2224"): 4,
    ("feathers_combined_uncorrected", "dmrn",   "J1744-1134"): 12,
    ("feathers_combined_uncorrected", "rnonly", "J1744-1134"): 12,

    ("feathers_lofaronly_corrected", "rnonly", "J1012+5307"): 4,
    ("feathers_lofaronly_corrected", "dmrn",   "J1012+5307"): 7,
    ("feathers_lofaronly_corrected", "rnonly", "J1640+2224"): 4,
    ("feathers_lofaronly_corrected", "dmrn",   "J1640+2224"): 4,
    ("feathers_lofaronly_corrected", "rnonly", "J1744-1134"): 6,
    ("feathers_lofaronly_corrected", "dmrn",   "J1744-1134"): 6,

    ("feathers_lofaronly_uncorrected", "rnonly", "J1012+5307"): 4,
    ("feathers_lofaronly_uncorrected", "dmrn",   "J1012+5307"): 4,
    ("feathers_lofaronly_uncorrected", "rnonly", "J1640+2224"): 4,
    ("feathers_lofaronly_uncorrected", "dmrn",   "J1640+2224"): 4,
    ("feathers_lofaronly_uncorrected", "rnonly", "J1744-1134"): 6,
    ("feathers_lofaronly_uncorrected", "dmrn",   "J1744-1134"): 6,
})

# Two cache layers: _cache (in-memory, dies with the kernel) and CACHE_DIR
# (on disk, survives a kernel restart / "Run All") -- loading a run means
# reading every post-burn epoch's h5 file and reshaping into physical-space
# arrays, which is the slow part, not the corner-plotting itself. Keyed by
# (pulsar, dataset, model, burn_epochs, thin), so a changed BURN_EPOCHS
# value naturally invalidates its own cache file instead of serving a
# stale burn. NOT auto-invalidated if a noise run itself gets rerun with
# new samples (same burn_epochs value, different underlying h5 content) --
# delete CACHE_DIR by hand after that.
CACHE_DIR = ".corner_explorer_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
_cache = {}  # (pulsar, dataset, model, thin) -> loaded data dict or None, in-memory only

def _cache_path(pulsar, case, thin, burn_epochs):
    dataset, model = case
    return os.path.join(CACHE_DIR, f"{pulsar}__{dataset}__{model}__burn{burn_epochs}__thin{thin}.npz")

def get_data(pulsar, case, thin):
    key = (pulsar,) + case + (thin,)
    if key in _cache:
        return _cache[key]

    dataset, model = case
    be = BURN_EPOCHS.get((dataset, model, pulsar))
    if be is None:
        print(f"  [warn] no burn_epochs entry for {case} / {pulsar} -- "
              f"using 4 as a fallback, verify against its trace before trusting this plot")
        be = 4

    cpath = _cache_path(pulsar, case, thin, be)
    if os.path.exists(cpath):
        npz = np.load(cpath)
        data = None if "__missing__" in npz.files else {k: npz[k] for k in npz.files}
    else:
        data = load_run(SAMPLES_DIR_BASE, dataset, model, pulsar, be, 0, thin)
        np.savez(cpath, **({"__missing__": np.array([True])} if data is None else data))

    _cache[key] = data
    return data


def plot_corner(pulsar, cases, group, thin):
    """cases: list of (dataset, model) tuples, in the order they should be
    drawn/listed in the legend. Draws nothing and returns quietly if none
    of them have a '{group}' parameter group (e.g. group="dmgp" with only
    rnonly cases selected -- rnonly runs have no DM GP, that is expected)."""
    loaded = {}
    for case in cases:
        data = get_data(pulsar, case, thin)
        if data is None:
            print(f"  [skip] {case} -- no data for {pulsar} (run not found)")
            continue
        loaded[case] = data

    names = None
    run_keys = [c for c in cases if c in loaded
                and any(group_of(n)[0] == group for n in loaded[c])]
    for c in run_keys:
        names_here = sorted(n for n in loaded[c] if group_of(n)[0] == group)
        names = names_here if names is None else names

    if not run_keys:
        print(f"no selected case has a '{group}' group for {pulsar} "
              f"(rnonly runs have no dmgp -- expected, not a bug)")
        return

    fig = None
    for case in run_keys:
        samples = np.vstack([loaded[case][n] for n in names]).T
        color, _ = CASE_STYLE[case]
        if fig is None:
            fig = corner_module.corner(samples, labels=names, color=color,
                                        show_titles=False, plot_datapoints=False,
                                        hist_kwargs=dict(density=True))
        else:
            corner_module.corner(samples, fig=fig, color=color,
                                  show_titles=False, plot_datapoints=False,
                                  hist_kwargs=dict(density=True))

    handles = [Line2D([0], [0], color=CASE_STYLE[c][0], lw=3, label=CASE_STYLE[c][1])
               for c in run_keys]
    fig.subplots_adjust(top=0.82)
    fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.98, 0.89),
               frameon=True, fontsize=11)
    fig.suptitle(f"{pulsar}: {group}", fontsize=13, fontweight="bold", y=0.97)
    plt.show()

## Available cases

The exact `(dataset, model)` tuples `CASES_ON` below can use -- copy/paste from here.

In [ ]:
for case, (color, label) in CASE_STYLE.items():
    print(f"{case!r:60s} {label}")

## Pick what to overlay

Edit `PULSAR` and `CASES_ON` (comment/uncomment or add/remove tuples -- order controls legend
order), then re-run this cell. Two corner plots are produced: `rednoise` always, `dmgp` only for
whichever selected cases have one (`dmrn` runs).

In [ ]:
PULSAR = "J1012+5307"
THIN = 20

CASES_ON = [
    ("feathers_combined_corrected", "dmrn"),
    ("feathers_highfreq_corrected", "dmrn"),
    # ("feathers_combined_uncorrected", "dmrn"),
    # ("feathers_lofaronly_corrected", "dmrn"),
    # ("feathers_lofaronly_uncorrected", "dmrn"),
    # ("feathers_combined_corrected", "rnonly"),
    # ("feathers_highfreq_corrected", "rnonly"),
    # ("feathers_combined_uncorrected", "rnonly"),
    # ("feathers_highfreq_uncorrected", "rnonly"),
    # ("feathers_lofaronly_corrected", "rnonly"),
    # ("feathers_lofaronly_uncorrected", "rnonly"),
]

for group in ["rednoise", "dmgp"]:
    plot_corner(PULSAR, CASES_ON, group, THIN)

## Noise spectra: pre- vs post-correction

Converts each posterior sample's `(log10_A, gamma)` into a powerlaw PSD curve $S(f)$, using
`discovery_new`'s own `powerlaw()` formula (read directly from
`epta-dr3/discovery_new/signals.py`, not assumed):
$$S(f) = \frac{A^2}{12\pi^2} (f/f_{yr})^{-\gamma} f_{yr}^{-3} \quad [s^3]$$
`dmgp` uses this exact same PSD function as `rednoise` (`model_builder.py`'s PSD table is generic
across GPs) -- its basis functions are scaled by $(f_{ref}/f_{obs})^2$ ($f_{ref}=1400$ MHz, same
convention as `compare_dm_timeseries.py`), so its recovered $A$/$\gamma$ parametrize the **delay
PSD at 1400 MHz**, not a frequency-independent quantity -- read its panel as "how much delay this
would cause at 1400 MHz", not absolute achromatic power.

The dashed horizontal line in each panel is a single approximate white-noise floor,
$S_{wn} = 2\langle\Delta t\rangle\langle\sigma^2\rangle$ (the standard sampled-white-noise PSD), with
$\langle\sigma^2\rangle$ an $n_\mathrm{TOA}$-weighted average across backends of
$(\mathrm{efac}\times\mathrm{median}(\sigma_\mathrm{TOA}))^2 + 10^{2\,\log_{10}\mathrm{EQUAD}}$, each
efac/EQUAD taken as its own posterior median for that case -- one line per backend would be more
accurate but unreadable with up to 13 backends, so this is the single-representative-line
convention PTA papers typically show. ECORR is excluded (correlated across one epoch's sub-TOAs,
not part of the per-TOA diagonal variance). The frequency axis spans $[1/T_\mathrm{span}, N_\mathrm{comp}/T_\mathrm{span}]$ -- the actual Fourier grid each GP was fit on ($N_\mathrm{comp}=30$ for rednoise, $100$ for dmgp, this project's fixed convention), taken from each case's own feather (the six per-pulsar datasets are Tspan-matched by design, computed per case rather than assumed identical).

In [ ]:
FYR = 1 / (365.25 * 86400)  # 1/year, in Hz

def powerlaw_psd(f, log10_A, gamma):
    """S(f) [s^3] -- discovery_new.signals.powerlaw with no df factor (that
    function returns phi = S(f)*df, the per-Fourier-bin prior variance)."""
    return 10.0**(2 * log10_A) / (12 * np.pi**2) * FYR**(gamma - 3) * f**(-gamma)


def spectrum_band(gammas, log10_As, freqs, percentiles=(16, 50, 84)):
    """(len(percentiles), len(freqs)) array: the percentile of S(f) *at each
    frequency*, across the posterior -- not "S(f) of the percentile of
    gamma/A", which would not be a meaningful curve."""
    psd = powerlaw_psd(freqs[None, :], log10_As[:, None], gammas[:, None])
    return np.percentile(psd, percentiles, axis=0)


N_COMPONENTS = {"rednoise": 30, "dmgp": 100}  # Fourier components per GP (fixed project convention)

_toa_stats_cache = {}  # (pulsar, dataset) -> (per-backend {name: (n_toa, median_toaerr_s)}, dt_avg_s, tspan_s)

def load_toa_stats(pulsar, dataset):
    key = (pulsar, dataset)
    if key not in _toa_stats_cache:
        t = feather.read_table(os.path.join(dataset, pulsar), columns=["toas", "toaerrs", "backend_flags"])
        toas = t["toas"].to_numpy()
        toaerrs = t["toaerrs"].to_numpy()
        backends = t["backend_flags"].to_numpy(zero_copy_only=False)
        per_backend = {}
        for b in np.unique(backends):
            sel = backends == b
            per_backend[b] = (int(sel.sum()), float(np.median(toaerrs[sel])))
        # NOT the median successive gap between raw TOAs -- with ~20 backends
        # observing near-simultaneously, that is dominated by same-session
        # TOAs microseconds-to-minutes apart and comes out meaninglessly
        # small (checked: 1.7e-6 days for J1012+5307 combined_corrected).
        # Tspan / n_unique_observing_days is the actual cadence the S_wn
        # formula means (independent noise realizations, not sub-band
        # channels of the same observation).
        mjd = toas / 86400.0
        n_days = len(np.unique(np.floor(mjd)))
        tspan = (mjd.max() - mjd.min()) * 86400.0  # seconds
        dt_avg = tspan / n_days
        _toa_stats_cache[key] = (per_backend, dt_avg, tspan)
    return _toa_stats_cache[key]


def wn_floor(pulsar, case, data):
    """Single approximate white-noise PSD level [s^3] for this case -- see
    the markdown cell above for the formula and its approximations."""
    dataset, model = case
    per_backend, dt_avg, _ = load_toa_stats(pulsar, dataset)
    total_n = sum(n for n, _ in per_backend.values())
    var_sum = 0.0
    for backend, (n, med_err) in per_backend.items():
        efac_key, eq_key = f"{backend}_efac", f"{backend}_log10_tnequad"
        efac = np.median(data[efac_key]) if efac_key in data else 1.0
        var = (efac * med_err) ** 2
        if eq_key in data:
            var += 10 ** (2 * np.median(data[eq_key]))
        var_sum += n * var
    sigma2_eff = var_sum / total_n
    return 2 * dt_avg * sigma2_eff


def plot_spectra(pulsar, cases, group, thin, n_freq=200):
    """One panel, all selected cases overlaid (corrected and uncorrected
    together -- CASE_STYLE's color + the legend label already say which is
    which, no need to isolate them into separate panels). Median + 68% band
    per case, colored/labelled via CASE_STYLE as everywhere else. Frequency
    axis is [1/Tspan, N_COMPONENTS[group]/Tspan] -- the actual Fourier grid
    each GP was fit on -- taken per case from that case's own feather
    (Tspan-matched by design across a pulsar's 6 datasets, but not assumed
    identical here)."""
    fig, ax = plt.subplots(figsize=(8, 5.5))
    drawn = False

    for case in cases:
        dataset, model = case
        data = get_data(pulsar, case, thin)
        if data is None:
            print(f"  [skip] {case} -- no data for {pulsar} (run not found)")
            continue
        gkey, akey = f"{group}_gamma", f"{group}_log10_A"
        if gkey not in data or akey not in data:
            continue  # e.g. group="dmgp" on an rnonly case -- expected, not a bug

        _, _, tspan = load_toa_stats(pulsar, dataset)
        f_lo, f_hi = 1 / tspan, N_COMPONENTS[group] / tspan
        freqs = np.logspace(np.log10(f_lo), np.log10(f_hi), n_freq)

        lo, med, hi = spectrum_band(data[gkey], data[akey], freqs)
        color, label = CASE_STYLE[case]
        ax.plot(freqs, med, color=color, lw=1.6, label=label)
        ax.fill_between(freqs, lo, hi, color=color, alpha=0.25)
        ax.axhline(wn_floor(pulsar, case, data), color=color, lw=1.0, ls="--", alpha=0.7,
                   label=f"{label} \u2014 WN floor")
        drawn = True

    if not drawn:
        print(f"no selected case has a '{group}' group for {pulsar} "
              f"(rnonly runs have no dmgp -- expected, not a bug)")
        plt.close(fig)
        return

    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("frequency [Hz]")
    ax.set_ylabel(r"$S(f)$ [s$^3$]" + (" (delay @ 1400 MHz)" if group == "dmgp" else ""))
    ax.legend(fontsize=8, loc="best")
    ax.set_title(f"{pulsar}: {group} power spectrum ({N_COMPONENTS[group]} components)",
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    plt.show()

Edit `PULSAR` / `CASES_ON` (same variables, reused from above) and re-run -- same
on-disk cache, so this is fast once a case has already been loaded for a corner plot.

In [ ]:
PULSAR = "J1012+5307"
THIN = 20

CASES_ON = [
    ("feathers_combined_corrected", "dmrn"),
    ("feathers_combined_uncorrected", "dmrn"),
]

for group in ["rednoise", "dmgp"]:
    plot_spectra(PULSAR, CASES_ON, group, THIN)